# Step 5b: RAG Framework — Retrieval, LLM & Agent
## YouTube Fast Fashion Intelligence Engine — CSCI370

### What this notebook covers

| Component | What it does |
|---|---|
| Metadata Retrieval | Filter comments by sentiment, video, topic |
| Semantic Retrieval | Find comments by meaning similarity |
| Lexical Retrieval (BM25) | Find comments by exact keyword match |
| Hybrid Retrieval | Combine semantic + lexical via RRF |
| Query Analysis | Understand what the user is asking |
| LLM Q&A | Answer questions grounded in retrieved comments |
| LLM Summarization | Summarize a set of retrieved comments |
| Agent Orchestration | Route each query to the right tool |

### End to end flow
```
User question -> Query Analysis -> Route to retrieval method
-> Retrieve top N comments -> Pass to Groq LLaMA 3 -> Grounded answer
```


## 0. Install Libraries

In [6]:
!pip install chromadb rank_bm25 groq scikit-learn -q

In [7]:
# ⚠️  IMPORTANT: Only run this cell if you have NOT already run step5a_rag_indexing.ipynb
# If you already built the ChromaDB database in Step 5a, SKIP THIS CELL entirely
# and go straight to Cell 5 (loading). Running this cell will wipe and rebuild
# the entire database which takes several minutes unnecessarily.

from chromadb.utils.embedding_functions import EmbeddingFunction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pickle
import chromadb
import pandas as pd

class TFIDFEmbeddingFunction(EmbeddingFunction):
    def __init__(self, fit_docs, n_components=128):
        self.tfidf = TfidfVectorizer(max_features=8000, stop_words='english',
                                     ngram_range=(1, 2), min_df=2)
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        X = self.tfidf.fit_transform(fit_docs)
        self.svd.fit(X)
    def __call__(self, input):
        X = self.tfidf.transform(input)
        return self.svd.transform(X).tolist()

df = pd.read_csv('youtube_comments_topics.csv', on_bad_lines='skip')
df = df.dropna(subset=['text_clean']).reset_index(drop=True)

with open('embedding_function.pkl', 'rb') as f:
    embedding_fn = pickle.load(f)

client = chromadb.PersistentClient(path='./chroma_db')
try:
    client.delete_collection('youtube_comments')
except:
    pass

collection = client.create_collection('youtube_comments', embedding_function=embedding_fn)

batch_size = 500
for start in range(0, len(df), batch_size):
    batch = df.iloc[start:start+batch_size]
    collection.add(
        documents=batch['text_clean'].tolist(),
        ids=[str(i) for i in batch.index],
        metadatas=[{
            'sentiment_label': str(row.get('sentiment_label', '')),
            'video_id'       : str(row.get('video_id', '')),
            'topic_general'  : int(row.get('topic_general', -1)),
            'like_count'     : int(row.get('like_count', 0)),
        } for _, row in batch.iterrows()]
    )
    if start % 5000 == 0:
        print(f"Indexed {start}/{len(df)}")

print(f"Done! Collection has {collection.count():,} documents")

Indexed 0/30609
Indexed 5000/30609
Indexed 10000/30609
Indexed 15000/30609
Indexed 20000/30609
Indexed 25000/30609
Indexed 30000/30609
Done! Collection has 30,609 documents


## 1. Load Everything from Step 5a

In [8]:
from chromadb.utils.embedding_functions import EmbeddingFunction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# This class must be defined here so Python can unpickle the embedding_function.pkl
# that was saved in step5a — pickle needs the class to exist in the current session.
class TFIDFEmbeddingFunction(EmbeddingFunction):
    def __init__(self, fit_docs, n_components=128):
        self.tfidf = TfidfVectorizer(max_features=8000, stop_words="english",
                                     ngram_range=(1, 2), min_df=2)
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        X = self.tfidf.fit_transform(fit_docs)
        self.svd.fit(X)

    def __call__(self, input):
        X = self.tfidf.transform(input)
        return self.svd.transform(X).tolist()

import pandas as pd
import numpy as np
import chromadb
import pickle
from groq import Groq
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('youtube_comments_topics.csv', on_bad_lines='skip')
df = df.dropna(subset=['text_clean']).reset_index(drop=True)

with open('embedding_function.pkl', 'rb') as f:
    embedding_fn = pickle.load(f)

client = chromadb.PersistentClient(path='./chroma_db')
collection = client.get_collection('youtube_comments', embedding_function=embedding_fn)

with open('bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

bm25_index   = bm25_data['index']
bm25_texts   = bm25_data['texts']
bm25_indices = bm25_data['indices']

print(f"Dataset     : {len(df):,} comments")
print(f"ChromaDB    : {collection.count():,} documents")
print(f"BM25 corpus : {len(bm25_texts):,} documents")
print("All loaded!")

Dataset     : 30,609 comments
ChromaDB    : 30,609 documents
BM25 corpus : 30,609 documents
All loaded!


## 2. Set Up Groq LLM

**How to get your free Groq API key:**
1. Go to https://console.groq.com
2. Sign up (no credit card needed)
3. API Keys -> Create API Key -> paste below


In [9]:
# Get your free Groq API key at: https://console.groq.com
# On Colab: add a secret named GROQ_API_KEY via the key icon in the left sidebar

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    # Local Jupyter fallback
    GROQ_API_KEY = "your-groq-api-key-here"

groq_client = Groq(api_key=GROQ_API_KEY)

# Test connection with the correct Groq model name
test = groq_client.chat.completions.create(
    model="llama3-70b-8192",
    messages=[{"role": "user", "content": "Say Groq connected and nothing else."}],
    max_tokens=10
)
print(test.choices[0].message.content)

## 3. The Four Retrieval Methods

Each method has different strengths. The agent decides which to use per query.


### Method 1: Metadata Filtering
Filter by stored properties. No query needed.

In [10]:
def metadata_retrieval(sentiment=None, video_id=None, topic=None, n=10):
    # Filter comments by their metadata fields
    # Use when: 'show me all negative comments', 'comments from video X'
    where_clause = {}
    if sentiment:
        where_clause['sentiment_label'] = sentiment
    if video_id:
        where_clause['video_id'] = video_id
    if topic is not None:
        where_clause['topic_general'] = topic

    if not where_clause:
        return df.nlargest(n, 'like_count')[['text_clean','sentiment_label','like_count']].to_dict('records')

    results = collection.get(where=where_clause, limit=n)
    return [{'text': doc, **meta} for doc, meta in zip(results['documents'], results['metadatas'])]

# Test
print("METADATA RETRIEVAL - Negative comments:")
for r in metadata_retrieval(sentiment='Negative', n=3):
    print(f"  [{r.get('sentiment_label')}] {str(r.get('text', r.get('text_clean','')))[:120]}")

METADATA RETRIEVAL - Negative comments:
  [Negative] This doesn't surprise me since the entire history of capitalism from the 19th century to today is the SAME. To really un
  [Negative] Okay then why work for these factories.....or are you forced to be there to work?
  [Negative] Please do not be this ignorant. The US and other countries are called "first-world" for a reason.


### Method 2: Semantic Search
Find comments by MEANING similarity. Works even with different words.

In [11]:
def semantic_retrieval(query, n=10, sentiment=None):
    # Find comments similar in meaning to the query
    # Use when: 'what do people think about environmental damage?'
    kwargs = {'query_texts': [query], 'n_results': n}
    if sentiment:
        kwargs['where'] = {'sentiment_label': sentiment}

    results = collection.query(**kwargs)
    return [{'text': doc, **meta} for doc, meta in zip(results['documents'][0], results['metadatas'][0])]

# Test
print("SEMANTIC RETRIEVAL - 'worker exploitation and low wages':")
for r in semantic_retrieval("worker exploitation and low wages", n=3):
    print(f"  [{r['sentiment_label']}] {r['text'][:150]}")

SEMANTIC RETRIEVAL - 'worker exploitation and low wages':
  [Negative] Low wages !!!! 🥴
  [Negative] Low purcashing power perhaps.
  [Neutral] That's some fairly low-key anecdotal evidence.


### Method 3: Lexical Search (BM25)
Find comments by EXACT keywords. Best for brand names and specific terms.

In [12]:
def lexical_retrieval(query, n=10):
    # Find comments by exact keyword matching using BM25
    # Use when: 'find comments about Temu', 'H&M specifically'
    # BM25 ranks by how often and how uniquely the query words appear
    tokens = query.lower().split()
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:n]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            orig_idx = bm25_indices[idx]
            row = df.iloc[orig_idx]
            results.append({
                'text'           : bm25_texts[idx],
                'bm25_score'     : float(scores[idx]),
                'sentiment_label': str(row.get('sentiment_label', '')),
                'like_count'     : int(row.get('like_count', 0)),
            })
    return results

# Test
print("LEXICAL RETRIEVAL - 'Temu cheap prices app':")
for r in lexical_retrieval("Temu cheap prices app", n=3):
    print(f"  [BM25: {r['bm25_score']:.2f}] [{r['sentiment_label']}] {r['text'][:150]}")

LEXICAL RETRIEVAL - 'Temu cheap prices app':
  [BM25: 16.23] [Negative] I deleted shein and temu app Thanx for video
  [BM25: 13.05] [Negative] my cheap ass ex did all her christmas shopping on this app dang
  [BM25: 12.56] [Positive] I was very suspi ious about shein and temu cheap prices. I go for higher quality natural fibers and make them last as long as possible . I mend . I ho


### Method 4: Hybrid Search
Combine semantic + lexical using **Reciprocal Rank Fusion (RRF)**.

RRF formula: score = sum(1 / (rank + 60)) for each list the document appears in.
Documents ranking well in BOTH lists get the highest scores.
This is the best general-purpose method.


In [13]:
def hybrid_retrieval(query, n=10, sentiment=None, semantic_weight=0.6, lexical_weight=0.4):
    # Best general method: combines semantic + lexical via RRF
    semantic_results = semantic_retrieval(query, n=n*2, sentiment=sentiment)
    lexical_results  = lexical_retrieval(query, n=n*2)

    rrf_scores = {}
    doc_data   = {}

    for rank, result in enumerate(semantic_results):
        key = result['text'][:100]
        rrf_scores[key] = rrf_scores.get(key, 0) + semantic_weight * (1 / (rank + 60))
        doc_data[key]   = result

    for rank, result in enumerate(lexical_results):
        key = result['text'][:100]
        rrf_scores[key] = rrf_scores.get(key, 0) + lexical_weight * (1 / (rank + 60))
        if key not in doc_data:
            doc_data[key] = result

    sorted_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:n]
    return [doc_data[k] for k in sorted_keys]

# Test
print("HYBRID RETRIEVAL - 'Shein environmental damage pollution':")
for r in hybrid_retrieval("Shein environmental damage pollution", n=3):
    print(f"  [{r.get('sentiment_label','?')}] {r.get('text','')[:150]}")

HYBRID RETRIEVAL - 'Shein environmental damage pollution':
  [Neutral] Shein is a tech firm - clue number 1
  [Negative] These celebrities should be ashamed of promoting SHEIN.
  [Negative] SHEIN also keeps stealing other artists designs and sells them as their own product


## 4. Query Analysis

Before retrieving, we analyze the query to decide:
- Which retrieval strategy to use
- Whether to apply a sentiment filter
- Whether to Q&A or summarize

| Question | Strategy |
|---|---|
| "What do negative comments say?" | Metadata |
| "Find comments about Temu" | Lexical |
| "How do people feel about the environment?" | Semantic |
| "What are the main concerns?" | Hybrid |
| "Summarize the discussion" | Hybrid + summarize |


In [14]:
def analyze_query(query):
    q = query.lower()

    # Detect sentiment filter
    sentiment_filter = None
    if any(w in q for w in ['negative','criticism','complain','hate','angry','upset','bad']):
        sentiment_filter = 'Negative'
    elif any(w in q for w in ['positive','praise','love','good','great','appreciate']):
        sentiment_filter = 'Positive'
    elif 'neutral' in q:
        sentiment_filter = 'Neutral'

    # Detect task
    task = 'summarize' if any(w in q for w in ['summarize','summary','overview','recap']) else 'qa'

    # Detect retrieval strategy
    brands = ['shein','temu','zara','h&m','primark','asos','boohoo','uniqlo','romwe']
    if any(b in q for b in brands) and len(q.split()) <= 5:
        strategy = 'lexical'
    elif any(w in q for w in ['feel','think','opinion','concern','worry','impact','environment']):
        strategy = 'semantic'
    else:
        strategy = 'hybrid'

    return {'strategy': strategy, 'sentiment_filter': sentiment_filter,
            'task': task, 'original_query': query}

# Test
test_queries = [
    "What do negative comments say about Shein?",
    "Summarize concerns about environmental damage",
    "Find comments about Temu",
    "How do people feel about fast fashion workers?",
]
print("QUERY ANALYSIS:")
print("-" * 60)
for q in test_queries:
    a = analyze_query(q)
    print(f"Query    : {q}")
    print(f"Strategy : {a['strategy']} | Sentiment: {a['sentiment_filter']} | Task: {a['task']}")
    print()

QUERY ANALYSIS:
------------------------------------------------------------
Query    : What do negative comments say about Shein?
Strategy : hybrid | Sentiment: Negative | Task: qa

Query    : Summarize concerns about environmental damage
Strategy : semantic | Sentiment: None | Task: summarize

Query    : Find comments about Temu
Strategy : lexical | Sentiment: None | Task: qa

Query    : How do people feel about fast fashion workers?
Strategy : semantic | Sentiment: None | Task: qa



## 5. LLM Q&A and Summarization

We pass retrieved comments to Groq LLaMA 3 as context.
The LLM is instructed to answer ONLY from those comments.
This prevents hallucination.


In [15]:
def llm_qa(query, retrieved_comments, max_comments=8):
    context = ""
    for i, c in enumerate(retrieved_comments[:max_comments], 1):
        text      = c.get('text', c.get('text_clean', ''))
        sentiment = c.get('sentiment_label', 'Unknown')
        likes     = c.get('like_count', 0)
        context  += f"[Comment {i} | Sentiment: {sentiment} | Likes: {likes}]\n{text}\n\n"

    prompt = (
        "You are an analyst studying YouTube comments about fast fashion brands.\n\n"
        "Answer the question below based ONLY on the YouTube comments provided.\n"
        "Do not use outside knowledge. Reference what commenters actually said.\n"
        "Keep your answer to 3-5 sentences.\n\n"
        f"QUESTION: {query}\n\n"
        f"YOUTUBE COMMENTS:\n{context}\nANSWER:"
    )

    response = groq_client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()


def llm_summarize(retrieved_comments, topic_description="fast fashion", max_comments=10):
    context = "".join([
        f"[Comment {i} | {c.get('sentiment_label','?')}]: {c.get('text', c.get('text_clean',''))}\n\n"
        for i, c in enumerate(retrieved_comments[:max_comments], 1)
    ])

    prompt = (
        f"Summarize these YouTube comments about {topic_description} in 5 sentences.\n"
        "Cover: main themes, overall sentiment, strongest concerns or praises, recurring patterns.\n\n"
        f"COMMENTS:\n{context}\nSUMMARY:"
    )

    response = groq_client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

print("LLM functions ready!")

LLM functions ready!


## 6. Agent Orchestration

The agent ties all components together:
1. Query analysis -> decide strategy and filters
2. Route to the right retrieval method
3. Call the LLM with retrieved context
4. Return grounded answer + source comments


In [16]:
def rag_agent(query, n_results=8):
    print(f"\nProcessing: '{query}'")
    print("-" * 60)

    analysis  = analyze_query(query)
    strategy  = analysis['strategy']
    sentiment = analysis['sentiment_filter']
    task      = analysis['task']

    print(f"Strategy: {strategy} | Sentiment: {sentiment} | Task: {task}")

    if strategy == 'lexical':
        retrieved = lexical_retrieval(query, n=n_results)
    elif strategy == 'semantic':
        retrieved = semantic_retrieval(query, n=n_results, sentiment=sentiment)
    elif strategy == 'metadata':
        retrieved = metadata_retrieval(sentiment=sentiment, n=n_results)
    else:
        retrieved = hybrid_retrieval(query, n=n_results, sentiment=sentiment)

    print(f"Retrieved: {len(retrieved)} comments")

    answer = llm_summarize(retrieved, topic_description=query) if task == 'summarize' else llm_qa(query, retrieved)

    return {
        'query'          : query,
        'analysis'       : analysis,
        'retrieved_count': len(retrieved),
        'answer'         : answer,
        'source_comments': retrieved[:3]
    }

print("RAG agent ready!")

RAG agent ready!


In [17]:
# Test the full agent
test_questions = [
    "What do people say about Shein's labor practices?",
    "Summarize the negative comments about fast fashion environmental impact",
    "How do positive commenters justify buying from Temu?",
]

for question in test_questions:
    result = rag_agent(question)
    print(f"\nQUESTION : {result['query']}")
    print(f"\nANSWER:")
    print(result['answer'])
    if result['source_comments']:
        src = result['source_comments'][0]
        print(f"\nTop source: [{src.get('sentiment_label','?')}] {src.get('text','')[:180]}")
    print("\n" + "="*65)


Processing: 'What do people say about Shein's labor practices?'
------------------------------------------------------------
Strategy: hybrid | Sentiment: None | Task: qa
Retrieved: 8 comments

QUESTION : What do people say about Shein's labor practices?

ANSWER:
The comments that mention Shein focus on a moral objection to the brand rather than specific details about its factories. One commenter says they will “never purchase anything from Shein,” indicating a personal boycott. Another criticizes “people… are so freaking disconnected from reality, and disconnected from what we consume,” suggesting that the brand’s practices—implicitly its labor conditions—are being ignored by the public. None of the other comments address Shein’s labor practices at all.

Top source: [Positive] I can proudly say I have never and will never purchase anything from Shein. Period.


Processing: 'Summarize the negative comments about fast fashion environmental impact'
--------------------------------------

## 7. Save RAG Pipeline for the Dashboard

In [18]:
rag_code = '''
import numpy as np
import chromadb
from chromadb import EmbeddingFunction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pickle
from groq import Groq

# Must be defined here so pickle can load embedding_function.pkl correctly
class TFIDFEmbeddingFunction(EmbeddingFunction):
    def __init__(self, fit_docs, n_components=128):
        self.tfidf = TfidfVectorizer(max_features=8000, stop_words="english",
                                     ngram_range=(1, 2), min_df=2)
        self.svd   = TruncatedSVD(n_components=n_components, random_state=42)
        X = self.tfidf.fit_transform(fit_docs)
        self.svd.fit(X)
    def __call__(self, input):
        X = self.tfidf.transform(input)
        return self.svd.transform(X).tolist()

def load_rag_pipeline(groq_api_key, db_path="./chroma_db",
                      embedding_pkl="embedding_function.pkl",
                      bm25_pkl="bm25_index.pkl"):
    with open(embedding_pkl, "rb") as f:
        embedding_fn = pickle.load(f)
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_collection("youtube_comments", embedding_function=embedding_fn)
    with open(bm25_pkl, "rb") as f:
        bm25_data = pickle.load(f)
    groq_client = Groq(api_key=groq_api_key)
    return collection, bm25_data, groq_client

def rag_query(query, collection, bm25_data, groq_client, n_results=8):
    bm25_index   = bm25_data["index"]
    bm25_texts   = bm25_data["texts"]
    bm25_indices = bm25_data["indices"]
    q = query.lower()

    sentiment_filter = None
    if any(w in q for w in ["negative","complain","hate","bad","angry"]):
        sentiment_filter = "Negative"
    elif any(w in q for w in ["positive","love","great","good","praise"]):
        sentiment_filter = "Positive"

    task = "summarize" if any(w in q for w in ["summarize","summary","overview"]) else "qa"

    brands = ["shein","temu","zara","h&m","primark","asos","boohoo"]
    if any(b in q for b in brands) and len(q.split()) <= 5:
        strategy = "lexical"
    elif any(w in q for w in ["feel","think","concern","impact","environment"]):
        strategy = "semantic"
    else:
        strategy = "hybrid"

    if strategy == "lexical":
        scores   = bm25_index.get_scores(q.split())
        top_idxs = np.argsort(scores)[::-1][:n_results]
        docs     = [{"text": bm25_texts[i], "sentiment_label": "", "like_count": 0}
                    for i in top_idxs if scores[i] > 0]
    elif strategy == "semantic":
        kwargs = {"query_texts": [query], "n_results": n_results}
        if sentiment_filter:
            kwargs["where"] = {"sentiment_label": sentiment_filter}
        res  = collection.query(**kwargs)
        docs = [{"text": d, **m} for d, m in zip(res["documents"][0], res["metadatas"][0])]
    else:
        kwargs = {"query_texts": [query], "n_results": n_results * 2}
        if sentiment_filter:
            kwargs["where"] = {"sentiment_label": sentiment_filter}
        sem      = collection.query(**kwargs)
        sem_docs = [{"text": d, **m} for d, m in zip(sem["documents"][0], sem["metadatas"][0])]
        lex_scrs = bm25_index.get_scores(q.split())
        lex_top  = np.argsort(lex_scrs)[::-1][:n_results * 2]
        lex_docs = [{"text": bm25_texts[i], "sentiment_label": "", "like_count": 0} for i in lex_top]
        seen, docs = set(), []
        for doc in sem_docs + lex_docs:
            key = doc["text"][:80]
            if key not in seen:
                seen.add(key)
                docs.append(doc)
            if len(docs) >= n_results:
                break

    context = "".join([
        f"[Comment {i} | {d.get(\'sentiment_label\',\'?\')}]: {d[\'text\']}\\n\\n"
        for i, d in enumerate(docs[:8], 1)
    ])

    if task == "summarize":
        prompt = f"Summarize these YouTube comments about fast fashion in 5 sentences:\\n\\n{context}\\nSUMMARY:"
    else:
        prompt = (
            f"Answer based ONLY on these comments. Be concise (3-5 sentences).\\n\\n"
            f"Question: {query}\\n\\nComments:\\n{context}\\nAnswer:"
        )

    resp = groq_client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.3
    )
    return {
        "answer"  : resp.choices[0].message.content.strip(),
        "sources" : docs[:3],
        "strategy": strategy,
        "task"    : task
    }
'''

with open('rag_pipeline.py', 'w') as f:
    f.write(rag_code)
print("rag_pipeline.py saved - dashboard will import this.")

rag_pipeline.py saved - dashboard will import this.


## Summary

**What we built:**

| Component | Function | Purpose |
|---|---|---|
| Metadata Retrieval | `metadata_retrieval()` | Filter by fields |
| Semantic Retrieval | `semantic_retrieval()` | Meaning similarity |
| Lexical Retrieval | `lexical_retrieval()` | Exact keywords (BM25) |
| Hybrid Retrieval | `hybrid_retrieval()` | Semantic + Lexical via RRF |
| Query Analysis | `analyze_query()` | Route to right tool |
| LLM Q&A | `llm_qa()` | Grounded answers |
| LLM Summarization | `llm_summarize()` | Comment summaries |
| Agent | `rag_agent()` | Orchestrates everything |

**Files produced:** `rag_pipeline.py`

**What to say in your report:**
*"A RAG pipeline was implemented using ChromaDB for vector storage and Groq LLaMA 3 (70B)
as the generation model. Four retrieval strategies were implemented: metadata filtering,
semantic search via TF-IDF embeddings, lexical search via BM25, and hybrid retrieval
using Reciprocal Rank Fusion (RRF). A query analysis module routes each user query to the
appropriate strategy, preventing hallucination by grounding all answers in real comments."*

**Next -> Step 6: Evaluation & MLflow**
